# 05 · Redes neuronales: del perceptrón al MLP entrenable

Este lab construye la intuición que después usaremos en visión, NLP, audio, transformers y modelos generativos.

## Objetivos
- Entender neurona, capa, activación, forward pass y función de pérdida.
- Ver por qué XOR exige no linealidad.
- Entender backpropagation como aplicación eficiente de la regla de la cadena.
- Comparar SGD y Adam.
- Practicar mini-batches, regularización L2, dropout y early stopping.
- Separar logits de probabilidades y elegir la loss correcta.


## 1. De una neurona a una red
Una neurona calcula $z=w^Tx+b$ y luego aplica una activación $a=\phi(z)$. Sin activaciones no lineales, apilar capas lineales sigue siendo una transformación lineal.

Activaciones comunes:
- **ReLU:** simple y eficiente, estándar en muchas redes.
- **GELU:** muy usada en transformers.
- **tanh:** salida entre -1 y 1, útil en ejemplos y algunas redes recurrentes.
- **sigmoid:** útil para probabilidad binaria en la salida, aunque `BCEWithLogitsLoss` es numéricamente más estable.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42); np.random.seed(42)
x=np.linspace(-5,5,400)
plt.plot(x,np.maximum(0,x),label='ReLU'); plt.plot(x,np.tanh(x),label='tanh'); plt.plot(x,1/(1+np.exp(-x)),label='sigmoid'); plt.legend(); plt.grid(); plt.show()

## 2. XOR demuestra la necesidad de capas ocultas
XOR no es linealmente separable. Un perceptrón de una sola frontera no puede resolverlo, pero un MLP con una capa oculta sí.


In [ ]:
X=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=torch.tensor([[0.],[1.],[1.],[0.]])
model=nn.Sequential(nn.Linear(2,8),nn.Tanh(),nn.Linear(8,1))
loss_fn=nn.BCEWithLogitsLoss(); opt=torch.optim.Adam(model.parameters(),lr=.05)
history=[]
for epoch in range(1200):
    logits=model(X); loss=loss_fn(logits,y)
    opt.zero_grad(); loss.backward(); opt.step(); history.append(loss.item())
print('probabilidades:',torch.sigmoid(model(X)).detach().ravel().numpy().round(3))
plt.plot(history); plt.yscale('log'); plt.title('Pérdida XOR'); plt.show()

## 3. ¿Qué hace backpropagation?
Si la pérdida es $L$ y una red tiene parámetros $\theta$, necesitamos $\partial L/\partial \theta$. Autodiff construye un grafo de operaciones y aplica la regla de la cadena desde la salida hacia atrás.

El optimizador usa esos gradientes. SGD actualiza $\theta \leftarrow \theta-\eta\nabla L$. Adam mantiene medias móviles de gradientes y cuadrados de gradientes, adaptando la escala de actualización.


In [ ]:
# Inspeccionar gradientes de una pasada
model.zero_grad(); logits=model(X); loss=loss_fn(logits,y); loss.backward()
for name,p in model.named_parameters():
    print(name,'shape=',tuple(p.shape),'grad_norm=',round(float(p.grad.norm()),6))

## 4. Ejemplo real: clasificación con mini-batches
Usaremos un dataset tabular. Para deep learning real el principio es el mismo, aunque imágenes/texto usan arquitecturas más especializadas.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
Xn,yn=load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(Xn,yn,test_size=.2,random_state=42,stratify=yn)
sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
train_ds=TensorDataset(torch.tensor(Xtr,dtype=torch.float32),torch.tensor(ytr[:,None],dtype=torch.float32))
loader=DataLoader(train_ds,batch_size=32,shuffle=True)
net=nn.Sequential(nn.Linear(Xtr.shape[1],64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
opt=torch.optim.AdamW(net.parameters(),lr=1e-3,weight_decay=1e-4); criterion=nn.BCEWithLogitsLoss()
losses=[]
for epoch in range(80):
    net.train(); total=0
    for xb,yb in loader:
        loss=criterion(net(xb),yb); opt.zero_grad(); loss.backward(); opt.step(); total+=loss.item()*len(xb)
    losses.append(total/len(train_ds))
plt.plot(losses); plt.title('training loss'); plt.show()
net.eval(); proba=torch.sigmoid(net(torch.tensor(Xte,dtype=torch.float32))).detach().numpy().ravel(); pred=(proba>=.5).astype(int)
print('ROC-AUC',roc_auc_score(yte,proba)); print(classification_report(yte,pred,digits=3))

## 5. Regularización y generalización
- **Weight decay/L2:** penaliza pesos grandes.
- **Dropout:** apaga unidades aleatoriamente durante entrenamiento.
- **Early stopping:** detiene cuando validation deja de mejorar.
- **Batch normalization:** estabiliza distribuciones internas; frecuente en CNN.
- **Data augmentation:** crea variaciones de entradas; clave en visión/audio.
- **Label smoothing:** evita predicciones excesivamente confiadas en clasificación multiclase.

## 6. Learning rate importa más de lo que parece
Si es demasiado alto, la loss oscila/diverge; si es muy bajo, converge lentamente o queda en regiones pobres. Schedulers, warmup y cosine decay son comunes en modelos grandes.


In [ ]:
# Mini-experimento: learning rates distintos en XOR
def train_lr(lr):
    m=nn.Sequential(nn.Linear(2,8),nn.ReLU(),nn.Linear(8,1)); o=torch.optim.SGD(m.parameters(),lr=lr); hist=[]
    for _ in range(400):
        l=loss_fn(m(X),y); o.zero_grad(); l.backward(); o.step(); hist.append(l.item())
    return hist
for lr in [.001,.05,.5]: plt.plot(train_lr(lr),label=str(lr))
plt.legend(title='learning rate'); plt.yscale('log'); plt.show()

## Cuándo una MLP es apropiada
- datos tabulares como baseline neuronal;
- embeddings ya calculados;
- capas finales de modelos multimodales;
- aprender relaciones no lineales cuando hay suficientes datos.

En tabular, gradient boosting suele competir o superar a MLP con menos tuning. Deep learning brilla especialmente en imagen, texto, audio, video y grandes representaciones.

## Errores comunes
- aplicar sigmoid y además `BCEWithLogitsLoss` (doble sigmoid);
- olvidar `model.eval()` para dropout/batchnorm;
- normalizar test con estadísticas de test;
- entrenar sin validation;
- aumentar capas sin mirar datos/overfitting;
- confundir epochs con pasos de optimización.

## Ejercicios
1. Implementa una neurona y MSE solo con NumPy.
2. Deriva manualmente el gradiente para regresión lineal.
3. Compara SGD, SGD+momentum, Adam y AdamW.
4. Añade un validation split y early stopping.
5. Compara ReLU, tanh y GELU.
6. Crea una curva de learning rate vs validation score.
7. Usa TensorBoard o MLflow para registrar experimentos.
